In [ ]:
%pip install -q krauncher-jupyter

# Tutorial 21: LoRA fine-tune Qwen2.5-7B with `%%krauncher`

The trendy one: parameter-efficient fine-tuning of a 7B LLM. This adapts
`Qwen/Qwen2.5-7B` with LoRA adapters on a subset of the Alpaca instruction
dataset, then has the fine-tuned model answer an unseen instruction — all from
a single marked notebook cell.

The cell is written the ordinary way: literal `from_pretrained("Qwen/Qwen2.5-7B")`
and `load_dataset("tatsu-lab/alpaca")`. The magic pre-fetches those Hub
references (the ~15 GB base model and the dataset) into the worker's cache
*before* the container starts, runs the cell on the cheapest GPU that fits,
streams a wall-clock training heartbeat into the cell, and returns the metrics
and a sample generation as notebook variables.

**Requirements:** this needs a **24 GB** GPU and takes roughly **20-30 min**
(base-model download + training). Only LoRA adapters are trained; the base
weights stay frozen in fp16.

Because each marked cell is one ephemeral task with no GPU state carried to the
next cell, training and the sample generation happen together in the one cell —
the fine-tuned adapter is used for inference before the worker is torn down.

In [ ]:
%load_ext krauncher_magic

In [ ]:
import os, getpass

os.environ["CAS_API_KEY"] = getpass.getpass("CAS_API_KEY: ")

### Fine-tune config

These become the marked cell's inputs (auto-detected as its free variables). `num_samples` caps the Alpaca subset; drop it for a longer, higher-quality run.

In [ ]:
num_samples = 3000
num_epochs = 3
batch_size = 4
grad_accum = 4
max_seq_len = 512
lora_r = 16
lora_alpha = 32
lr = 2e-4

### Train + generate

Run the cell — the quote prints first, then a training heartbeat, then the fine-tuned model's answer to an unseen instruction. The metrics and the sample response come back as notebook variables.

In [ ]:
%%krauncher --pip peft,datasets --timeout 1800 --out train_loss,train_runtime_sec,train_samples,effective_batch,sample_response
# --timeout (on the line above) is a safety cap on the remote run: if the
# code hangs, the host is shut down once that many seconds elapse instead of
# billing forever — it protects your spend.
import time
import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

# HF-native: the literal Hub refs below are pre-fetched by the data bridge
# before the container starts — Qwen2.5-7B (~15 GB) and Alpaca land in the
# local hub cache, so from_pretrained / load_dataset resolve offline.
print("Loading tokenizer...", flush=True)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model weights (fp16, ~14 GB)...", flush=True)
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B", torch_dtype=torch.float16, device_map="auto",
)
model.config.use_cache = False

lora_cfg = LoraConfig(
    r=lora_r, lora_alpha=lora_alpha, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type=TaskType.CAUSAL_LM, bias="none",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

ds_full = load_dataset("tatsu-lab/alpaca")["train"]
ds = ds_full.shuffle(seed=42).select(range(min(num_samples, len(ds_full))))

def format_alpaca(ex):
    if ex.get("input"):
        text = (f"### Instruction:\n{ex['instruction']}\n\n"
                f"### Input:\n{ex['input']}\n\n"
                f"### Response:\n{ex['output']}{tokenizer.eos_token}")
    else:
        text = (f"### Instruction:\n{ex['instruction']}\n\n"
                f"### Response:\n{ex['output']}{tokenizer.eos_token}")
    return tokenizer(text, truncation=True, max_length=max_seq_len, padding="max_length")

print(f"Tokenizing {len(ds)} samples...", flush=True)
ds = ds.map(format_alpaca, remove_columns=ds.column_names)
ds.set_format("torch")

args = TrainingArguments(
    output_dir="/tmp/qwen-lora",
    num_train_epochs=num_epochs,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    learning_rate=lr,
    weight_decay=0.0,
    fp16=True,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    dataloader_num_workers=2,
    optim="adamw_torch",
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

class Heartbeat(TrainerCallback):
    """Wall-clock heartbeat so progress shows even if a step is slow."""
    def __init__(self):
        self._t0 = time.monotonic()
        self._last = self._t0
    def on_step_end(self, args, state, control, **kwargs):
        now = time.monotonic()
        if now - self._last >= 50:
            loss = state.log_history[-1].get("loss") if state.log_history else None
            loss_str = f"loss={loss:.4f}" if loss is not None else "loss=n/a"
            print(f"  [step {state.global_step}/{state.max_steps}] {loss_str} "
                  f"(elapsed {now - self._t0:.0f}s)", flush=True)
            self._last = now

trainer = Trainer(
    model=model, args=args, train_dataset=ds,
    data_collator=collator, callbacks=[Heartbeat()],
)
print(f"Starting LoRA training: epochs={num_epochs}, "
      f"batch={batch_size}x{grad_accum}, r={lora_r}", flush=True)
train_result = trainer.train()

train_loss = round(train_result.training_loss, 4)
train_runtime_sec = round(train_result.metrics["train_runtime"], 1)
train_samples = len(ds)
effective_batch = batch_size * grad_accum
print(f"Training done: loss={train_loss}, runtime={train_runtime_sec:.0f}s", flush=True)

# Use the fine-tuned adapter to answer an unseen instruction (per-cell model).
print("Generating a sample response from the fine-tuned adapter...", flush=True)
model.config.use_cache = True
prompt = ("### Instruction:\nExplain what LoRA fine-tuning is in two sentences."
          "\n\n### Response:\n")
gen_inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    gen = model.generate(
        **gen_inputs, max_new_tokens=120, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
sample_response = tokenizer.decode(
    gen[0][gen_inputs["input_ids"].shape[1]:], skip_special_tokens=True,
).strip()
print("Sample response:", sample_response, flush=True)

### Results

In [ ]:
print(f"Train loss:      {train_loss}")
print(f"Train samples:   {train_samples}")
print(f"Effective batch: {effective_batch}")
print(f"LoRA rank:       {lora_r}")
print(f"Train runtime:   {train_runtime_sec:.0f}s")
print(f"\nSample response from the fine-tuned model:\n{sample_response}")